In [1]:
# Loading the RAG answers
import pandas as pd

df_answers = pd.read_csv("data/rag_results.csv")
answers = df_answers.to_dict(orient="records")

##### A -> Q -> A' Evaluation

In [2]:
# Define the output format
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(description="Reasoning about the quality of the answer.")
    score: Literal["Good", "Bad"] = Field(description="'Good' if the answer is correct and complete, 'Bad' otherwise.")

In [3]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical.
- It should convey the same key information.
- Extra detail is fine as long as the core answer is correct.
- Mark 'Bad' only if the AI answer is wrong or misses the key point.

CRITICAL INSTRUCTIONS FOR OUTPUT FORMAT:
1. You MUST always provide a detailed "reasoning" string explaining your thought process. Do not leave it empty.
2. The "score" MUST BE EXACTLY ONE OF THESE TWO WORDS: "Good" or "Bad". 
3. DO NOT use words like "Partial", "Partial match", or "Average". ONLY "Good" or "Bad".

Your output must strictly follow this JSON format:
{
    "reasoning": "your detailed explanation...",
    "score": "Good"
}
""".strip()

In [4]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [5]:
# Import the structured output helper
from openai import OpenAI
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

openai_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama"  
)

In [6]:
# Take one record
rec = answers[0]

In [7]:
# Create the judge prompt
judge_prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_original"],
    answer_llm=rec["answer_llm"]
)

In [8]:
judge_prompt

'Question:\nIf I miss the deadline for submissions, can I still get a certificate?\n\nOriginal Answer (ground truth):\nYes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.\n\nAI Answer:\nTo answer your question directly: \n\nYes, you CAN still get a certificate.\n\nThe text does not mention missing a homework submission, but getting an extension for one is mentioned in this section : A: "No. We don\'t give individual deadline extensions, and once the homework submission form is closed you can no longer submit it — there are no late submissions."\n\nHowever the exact question asked is If I MISS THE DEADLINE FOR SUBMISSIONS...'

In [9]:
# Call the judge
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    user_prompt=judge_prompt,
    output_type=AnswerEvaluation
)

eval_result   

AnswerEvaluation(reasoning="The AI answer seems to conflate two different scenarios. The original answer explicitly states that missing a deadline for submissions does not necessarily prevent a certificate from being obtained, as long as the project is submitted while the submission window remains open. However, it distinguishes between submitting homework after the deadline (as in section A) and being eligible for certification at a later time. In contrast, the AI answer implies that missing a deadline entirely means one can still receive a certificate, which is not explicitly stated in the original answer. While a certificate does depend on submission while 'still accepting submissions,' a missed submission altogether does not guarantee it.", score='Bad')

In [10]:
calc_price(usage)

{'input_cost': 0.00029475,
 'output_cost': 0.0006345000000000001,
 'total_cost': 0.0009292500000000001}

In [11]:
# Put this logic into a function
def evaluate_aqa(question, answer_orig, answer_llm, model="llama3.2"):
    judge_prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )
    
    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        user_prompt=judge_prompt,
        output_type=AnswerEvaluation,
        model=model
    )
    
    return result, usage

In [12]:
# Test it on the same record
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_original"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning="The AI answer contains an unnecessary emphasis on words 'CAN', which conveys certainty that might not be present in the original context. The original ground truth cautions that receiving a certificate requires submitting the project while submissions are still open, but it does not guarantee that missing a deadline will result in automatic certificate issuance. Despite this, the core implication of the question being partially addressed in the AI answer is correct: that missing a submission deadline does affect eligibility for a certificate.", score='Bad')

In [13]:
# Run the evaluation on all records
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_original"],
        answer_llm=rec["answer_llm"]
    )
    
    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning
    }
    
    return result, usage

In [14]:
# Use the parallel processing helper to evaluate all records
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=4) as executor: 
    results = map_progress(executor, answers, judge_record)

  0%|          | 0/140 [00:00<?, ?it/s]

In [15]:
# Split the results into evaluations and usage
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [16]:
# Create a DataFrame for the evaluations
df_evaluations = pd.DataFrame(evaluations)

In [18]:
calc_total_price(usages)

0.12452324999999997

In [20]:
df_evaluations.head()

,question,document,score,reasoning
0,"If I miss the deadline for submissions, can I ...",74eb249bbf,Bad,The AI answer does not convey additional conte...
1,What is the format of the project submissions ...,74eb249bbf,Bad,This AI answer does not address the student's ...
2,Can someone who has already joined the course ...,74eb249bbf,Good,Both answers convey that someone who has joine...
3,How long after submitting my project will I re...,74eb249bbf,Bad,The AI answer captures the necessity of submis...
4,Will there be any penalties or late fees for m...,74eb249bbf,Bad,"The AI answer conveys some correct points, but..."


In [21]:
df_evaluations.score.value_counts()

score
Bad     90
Good    50
Name: count, dtype: int64

In [22]:
df_evaluations.score.value_counts(normalize=True)

score
Bad     0.642857
Good    0.357143
Name: proportion, dtype: float64

In [23]:
df_evaluations[df_evaluations["score"] == "Bad"].head()

,question,document,score,reasoning
0,"If I miss the deadline for submissions, can I ...",74eb249bbf,Bad,The AI answer does not convey additional conte...
1,What is the format of the project submissions ...,74eb249bbf,Bad,This AI answer does not address the student's ...
3,How long after submitting my project will I re...,74eb249bbf,Bad,The AI answer captures the necessity of submis...
4,Will there be any penalties or late fees for m...,74eb249bbf,Bad,"The AI answer conveys some correct points, but..."
5,"Hey, I've signed up for the LLM Zoomcamp. What...",977bf7786c,Bad,The AI attempt to summarize the original inten...


In [24]:
# Save the results to a CSV file
df_evaluations.to_csv("data/rag_evaluation_results.csv", index=False)